In [77]:
import pandas as pd
from datasets import load_from_disk
from IPython.display import HTML, display
import re
import io
import dis
from datasets import load_dataset
import os

# 1. Load your dataset and convert to Pandas
dataset = load_from_disk('./test_dataset')
ogdataset = dataset.to_pandas()
df = ogdataset[ogdataset['status'] != "Success"]


In [78]:
counts = df['status'].value_counts()

print(counts)

status
Extraction Error    492422
SyntaxError            463
Name: count, dtype: int64


In [80]:
work_df = df[df['status'] == "Extraction Error"]
i = 3
extracted_code = work_df['extracted_code'].iloc[i]
print(extracted_code)

No match found


In [81]:
status = work_df['status'].iloc[i]
print(status)

try:
    # Compile to verify valid Python syntax
    compiled_code = compile(extracted_code, '<string>', 'exec')
    
    # Disassemble into a readable bytecode string
    bytecode_io = io.StringIO()
    dis.dis(compiled_code, file=bytecode_io)
    bytecode_str = bytecode_io.getvalue()
    
    print("Success")
    
except Exception as e:
    print(f"Unexpected Error: {str(e)}")

Extraction Error
Unexpected Error: invalid syntax (<string>, line 1)


In [ ]:
PRIMARY_PATTERN = re.compile(r'\`\`\`python(.*?)\`\`\`', flags=re.DOTALL | re.IGNORECASE)
FALLBACK_PATTERN = re.compile(r'Here is the implementation.*?\:\s*(.*?)', flags=re.DOTALL | re.IGNORECASE)

text = work_df['output'].iloc[i]
print(text)
match = PRIMARY_PATTERN.search(text)

if match:
    # Found standard ```python code block
    extracted_code = match.group(1).strip()
    print(extracted_code)
else:
   new_match = FALLBACK_PATTERN.search(text)
   if new_match:
       extracted_code = new_match.group(1).strip()
       print(extracted_code)
   else:
       print("No match")



No match
